In [2]:
import sys
print(sys.executable)


/Users/divyanshunagar/Desktop/RANKING_SYSTEM/venv/bin/python3


In [5]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader

In [6]:
dataset="msmarco"
data_path = util.download_and_unzip(
    "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/msmarco.zip",
    "data/raw"
)
corpus,queries,qrels=GenericDataLoader(data_path).load(split="dev")
print("DOCS:",len(corpus))
print("QUERIES:",len(queries))

data/raw/msmarco.zip: 100%|████████████████| 1.01G/1.01G [21:53<00:00, 824kiB/s]
100%|█████████████████████████████| 8841823/8841823 [00:16<00:00, 546484.37it/s]


DOCS: 8841823
QUERIES: 6980


In [7]:
relevant_doc_ids = set()

for qid, doc_dict in qrels.items():
    for doc_id in doc_dict:
        relevant_doc_ids.add(doc_id)

print("Relevant docs (from qrels):", len(relevant_doc_ids))


Relevant docs (from qrels): 7433


In [8]:
filtered_corpus = {
    doc_id: corpus[doc_id]
    for doc_id in relevant_doc_ids
    if doc_id in corpus
}

print("Filtered corpus size:", len(filtered_corpus))


Filtered corpus size: 7433


In [9]:
import random
random.seed(42)

sampled_qids = random.sample(list(queries.keys()), 300)
sampled_queries = {qid: queries[qid] for qid in sampled_qids}

print("Sampled queries:", len(sampled_queries))


Sampled queries: 300


In [10]:
from rank_bm25 import BM25Okapi

doc_ids = list(filtered_corpus.keys())
docs_text = [filtered_corpus[d]["text"] for d in doc_ids]
tokenized_docs = [d.lower().split() for d in docs_text]

bm25 = BM25Okapi(tokenized_docs)
print("BM25 index")


BM25 index


In [11]:
qid = sampled_qids[0]
query = sampled_queries[qid]

print("QUERY:", query)
print("\nTOP-5 RESULTS:\n")

scores = bm25.get_scores(query.lower().split())
ranked = sorted(zip(doc_ids, scores), key=lambda x: x[1], reverse=True)[:5]

for doc_id, score in ranked:
    print(f"SCORE: {score:.2f}")
    print(filtered_corpus[doc_id]["text"][:300])
    print("-"*80)


QUERY: who is ice p

TOP-5 RESULTS:

SCORE: 17.86
Jimmy Lewis, known publicly by his stage name Ice P, was the victim in Shark Attack! (Case #1 of Pacific Bay). Ice P featuring in Shark Attack!'s promotional image.
--------------------------------------------------------------------------------
SCORE: 17.40
Ice P was a famous reality TV personality, of African-American heritage, sported tattoos on both of his shoulders, usually wore a purple tank top with yellow streaks, sported a gold necklace with a dollar sign on it, and also wore a purple-yellow hat. Ice P had black hair, black eyes and sported a b
--------------------------------------------------------------------------------
SCORE: 15.90
The definition of intense is to a high degree, or a strong emotion. 1  If you really, really really hate ice cream, this is an example of when you have an intense hatred of ice cream. 2  A person who is always serious and talking about problems and emotional issues is an example of someone who i

In [12]:
from ranx import Qrels, Run, evaluate

def bm25_rank(query, k=10):
    scores = bm25.get_scores(query.lower().split())
    ranked = sorted(zip(doc_ids, scores), key=lambda x: x[1], reverse=True)
    return ranked[:k]

run_dict = {}
for qid in sampled_qids:
    ranked = bm25_rank(sampled_queries[qid], k=10)
    run_dict[qid] = {doc: float(score) for doc, score in ranked}

qrels_subset = {qid: qrels[qid] for qid in sampled_qids if qid in qrels}

metrics = evaluate(Qrels(qrels_subset), Run(run_dict), ["ndcg@10", "map"])
metrics


Matplotlib is building the font cache; this may take a moment.
/Users/divyanshunagar/Desktop/RANKING_SYSTEM/venv/lib/python3.9/site-packages/ranx/metrics/ndcg.py:72: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  scores[i] = _ndcg(qrels[i], run[i], k, rel_lvl, jarvelin)


{'ndcg@10': np.float64(0.683408827231414),
 'map': np.float64(0.6447050264550264)}

In [13]:
import json
from pathlib import Path

Path("experiments").mkdir(exist_ok=True)
with open("experiments/bm25_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

metrics


{'ndcg@10': np.float64(0.683408827231414),
 'map': np.float64(0.6447050264550264)}